# Field-gate reproduction
This notebook reproduces the frozen Anki field-design analysis reported in the manuscript. Raw review logs are not redistributed because the upstream dataset asks users not to redistribute them publicly.

Frozen design: benchmark commit 1053082, seed 20260817, 1,000 reduced-information collections, fixed 150-collection full-profile subsample.

In [ ]:
!pip -q install numpy scipy pandas scikit-learn matplotlib pyarrow huggingface_hub hf_xet fsrs-optimizer==6.5.0


## Authenticate to Hugging Face
Accept the access conditions for open-spaced-repetition/anki-revlogs-10k in your Hugging Face account before running the next cell.

In [ ]:
from huggingface_hub import login
login()


In [ ]:
from pathlib import Path
import subprocess
REPO = Path('/content/anchor-identifiability-spaced-repetition')
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/huynhanhkhiem-dms/anchor-identifiability-spaced-repetition.git',str(REPO)], check=True)
BENCH = REPO / 'data' / 'srs-benchmark'
if not BENCH.exists():
    subprocess.run(['git','clone','-q','https://github.com/open-spaced-repetition/srs-benchmark.git',str(BENCH)], check=True)
subprocess.run(['git','-C',str(BENCH),'checkout','-q','1053082'], check=True)
print(REPO)


In [ ]:
from huggingface_hub import snapshot_download
DATA = REPO / 'data' / 'anki-revlogs-10k'
snapshot_download(repo_id='open-spaced-repetition/anki-revlogs-10k', repo_type='dataset', local_dir=str(DATA))
print(DATA)


## Run the frozen gate
The command below uses the exact sample sizes and seed reported in the manuscript. The script refuses to issue a benchmark-matched verdict if the required fsrs-optimizer filters are unavailable.

In [ ]:
cmd = [
    'python', str(REPO/'code'/'gate_design_measure.py'),
    '--data', str(DATA),
    '--bench', str(BENCH),
    '--users', '1000',
    '--full-profile', '150',
    '--seed', '20260817',
    '--out', str(REPO/'results'/'gate_design_measure_rerun.json')
]
subprocess.run(cmd, check=True)


If a long full-profile run is interrupted, use analysis/run_full_profile_checkpoint.py with the reduced JSON and a persistent checkpoint path. Do not change the seed, sample sizes, thresholds, or selected population after inspecting results.